In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 20)

/Users/mironshoh/Desktop/workspaces/projects/personal/CineMatch/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
processed_dir = Path('../data/processed')
full_dir = Path('../data/full_dataset')
model_dir = Path('../models')
model_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# Load already-filtered ratings (created by data_prep.ipynb)
ratings_f = pd.read_csv(processed_dir / 'ratings_filtered.csv')
print('ratings_f:', ratings_f.shape)

# Load movie metadata for inference
movies = pd.read_csv(full_dir / 'movies.csv')
print('movies:', movies.shape)

ratings_f: (31498689, 6)
movies: (87585, 3)


In [4]:
# Encode user and movie indices
user_ids = ratings_f['userId'].unique()
movie_ids = ratings_f['movieId'].unique()

user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}
movie_id_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

ratings_f['user_idx'] = ratings_f['userId'].map(user_id_to_idx)
ratings_f['movie_idx'] = ratings_f['movieId'].map(movie_id_to_idx)

n_users = len(user_ids)
n_movies = len(movie_ids)
print(f'Users: {n_users}, Movies: {n_movies}')

Users: 200947, Movies: 16034


In [5]:
# Create implicit feedback: like = rating >= 4.0
likes = ratings_f[ratings_f['rating'] >= 4.0].copy()
print('Positive interactions (likes):', len(likes))

Positive interactions (likes): 15799204


In [6]:
# Build sparse user-item matrix from all likes
X = csr_matrix(
    (np.ones(len(likes), dtype=np.float32), (likes['user_idx'], likes['movie_idx'])),
    shape=(n_users, n_movies)
)

print('Matrix shape:', X.shape)
print('Non-zeros:', X.nnz)
print('Sparsity: {:.4f}%'.format(100 * X.nnz / (X.shape[0] * X.shape[1])))

Matrix shape: (200947, 16034)
Non-zeros: 15799204
Sparsity: 0.4904%


In [7]:
# Best config from hyperparameter grid search
BEST_FACTORS = 256
BEST_REG = 0.1
BEST_ALPHA = 20
ITERATIONS = 20

print(f'Training ALS: factors={BEST_FACTORS}, reg={BEST_REG}, alpha={BEST_ALPHA}, iterations={ITERATIONS}')

model = AlternatingLeastSquares(
    factors=BEST_FACTORS,
    regularization=BEST_REG,
    iterations=ITERATIONS,
    random_state=42
)

model.fit(X * BEST_ALPHA)

Training ALS: factors=256, reg=0.1, alpha=20, iterations=20


/Users/mironshoh/Desktop/workspaces/projects/personal/CineMatch/.venv/lib/python3.9/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [11:19<00:00, 33.98s/it]


In [ ]:
# Build reverse mapping: movie_idx -> movieId
idx_to_movie_id = {v: k for k, v in movie_id_to_idx.items()}

# Build movie metadata lookup (movieId -> title, genres)
movie_meta = movies.set_index('movieId')[['title', 'genres']].to_dict('index')

model_artifacts = {
    'model': model,
    'user_id_to_idx': user_id_to_idx,
    'movie_id_to_idx': movie_id_to_idx,
    'idx_to_movie_id': idx_to_movie_id,
    'movie_meta': movie_meta,
    'config': {
        'factors': BEST_FACTORS,
        'regularization': BEST_REG,
        'alpha': BEST_ALPHA,
        'iterations': ITERATIONS,
    }
}

model_path = model_dir / 'als_model_initial.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_artifacts, f)

print(f'Model + artifacts saved to {model_path}')
print(f'Model size: {model_path.stat().st_size / 1024 / 1024:.1f} MB')

Model + artifacts saved to ../models/als_model.pkl
Model size: 221.2 MB


In [9]:
# Quick sanity check: recommend for a sample user
sample_user_id = 1  # userId 1
if sample_user_id in user_id_to_idx:
    user_idx = user_id_to_idx[sample_user_id]
    recs = model.recommend(user_idx, X[user_idx], N=5, filter_already_liked_items=True)
    rec_ids = [int(r) if isinstance(r, (np.integer, int)) else int(r[0]) for r in recs[0]]
    rec_movie_ids = [idx_to_movie_id[idx] for idx in rec_ids]
    print(f'Top 5 recommendations for user {sample_user_id}:')
    for mid in rec_movie_ids:
        meta = movie_meta.get(mid, {})
        print(f'  {meta.get("title", "?")}  ({meta.get("genres", "?")})')
else:
    print(f'User {sample_user_id} not in filtered dataset')

Top 5 recommendations for user 1:
  Bridge on the River Kwai, The (1957)  (Adventure|Drama|War)
  Player, The (1992)  (Comedy|Crime|Drama)
  Annie Hall (1977)  (Comedy|Romance)
  Say Anything... (1989)  (Comedy|Drama|Romance)
  American Beauty (1999)  (Drama|Romance)
